# Synteny of the RBH hits

For every reciprocal best hit from `1_run_bidirectional_blast.py`, checks whether other hits of the same
reference cluster (SCO or VNZ) in the same genome lie within `N_NEIGHBOURS` genes on either side.

Gene order is taken from the protein `.faa` files (proteins are in genome order), via `shared/synteny.py`.
Positions run continuously through each file, so on draft genomes two genes at the ends of different contigs
can count as neighbours.

In [1]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_DIR = Path("/vol/local/calarass/Projects/ara_comp_genomics")
sys.path.insert(0, str(PROJECT_DIR / "shared"))
from synteny import find_neighbours, get_gene_iterator

RBH_HITS = PROJECT_DIR / "RBH" / "results" / "diamond_reverseBLAST_streptomycetae" / "reciprocal" / "reciprocal_hits.tsv"
FAA_DIR = Path("/vol/local/calarass/Projects/lipid_genomics/faa_files")
N_NEIGHBOURS = 5

## 1. Load the RBH hits

In [2]:
rbh_hits = pd.read_csv(RBH_HITS, sep="\t")
rbh_hits["locus_tag"] = rbh_hits.sseqid.str.split("|").str[1]

print(f"{len(rbh_hits)} hits, {rbh_hits.genome.nunique()} genomes")
rbh_hits.groupby(["ref_group", "qseqid"]).genome.nunique()

836 hits, 156 genomes


ref_group  qseqid        
SCO        SCO2401           130
           SCO2402            94
           SCO2403           129
           SCO2407            95
           SCO2439           125
           SCO2440           132
VNZ        vnz_33170-lacI     43
           vnz_33175-araB     32
           vnz_33180-araA     32
           vnz_33185-araD     24
Name: genome, dtype: int64

## 2. Match genomes to their `.faa` files

By the genome name in the first header, not the file name (file names are not consistent, e.g. no `_protein` suffix).

In [3]:
def genome_of_faa(faa_file: Path) -> str:
    """Genome name from the first header, brackets dropped to match the genome column of reciprocal_hits.tsv."""
    with open(faa_file) as f:
        header = f.readline()
    return header.lstrip(">").split("|")[0].replace("[", "").replace("]", "")


FAA_FILES = {genome_of_faa(path): path for path in sorted(FAA_DIR.glob("*.faa"))}

missing = sorted(set(rbh_hits.genome) - set(FAA_FILES))
if missing:
    print("WARNING: no .faa file for:", missing)

## 3. Neighbouring hits

Per reference group (SCO / VNZ) and genome: for each hit, which other hits of the same group are among its
`N_NEIGHBOURS` genes to the left and right.

In [4]:
rows = []
for genome, genome_hits in rbh_hits.groupby("genome"):
    gene_positions = get_gene_iterator(FAA_FILES[genome])
    for ref_group, group_hits in genome_hits.groupby("ref_group"):
        hit_genes = dict(zip(group_hits.locus_tag, group_hits.qseqid))
        for hit in group_hits.itertuples():
            neighbours = find_neighbours(hit.locus_tag, gene_positions, N_NEIGHBOURS)
            neighbour_hits = [f"{hit_genes[n]}:{n}" for n in neighbours if n in hit_genes]
            rows.append({
                "ref_group": ref_group,
                "genome": genome,
                "qseqid": hit.qseqid,
                "locus_tag": hit.locus_tag,
                "neighbour_hits": ", ".join(neighbour_hits),
                "has_neighbours": "yes" if neighbour_hits else "no",
            })

synteny_table = pd.DataFrame(rows).sort_values(["ref_group", "genome", "qseqid"]).reset_index(drop=True)
pd.crosstab([synteny_table.ref_group, synteny_table.qseqid], synteny_table.has_neighbours)

has_neighbours            no  yes
ref_group qseqid                 
SCO       SCO2401          0  130
          SCO2402          0   94
          SCO2403          0  129
          SCO2407          6   89
          SCO2439          6  119
          SCO2440          2  130
VNZ       vnz_33170-lacI  22   21
          vnz_33175-araB   1   31
          vnz_33180-araA   1   31
          vnz_33185-araD   2   22

In [5]:
synteny_table.loc[synteny_table.has_neighbours == "no"].groupby(["ref_group", "qseqid"]).genome.nunique().sort_values(ascending=False).head(10)

ref_group  qseqid        
VNZ        vnz_33170-lacI    22
SCO        SCO2439            6
           SCO2407            6
           SCO2440            2
VNZ        vnz_33185-araD     2
           vnz_33175-araB     1
           vnz_33180-araA     1
Name: genome, dtype: int64

### 3b. SCO2401 / SCO2402 / SCO2403 trio

In *S. coelicolor* SCO2401-SCO2403 sit next to each other. Per genome and query gene: is the gene directly to the
left or right (`TRIO_WINDOW` genes) of the hit itself a hit of another trio gene?

`trio_summary` then gives one row per genome: `trio` = all three genes in one adjacent block, `pair` = at most two
of them adjacent, `none` = hits present but not adjacent. Tandem copies of the same gene next to each other do not count.

In [12]:
TRIO = ["SCO2401", "SCO2402", "SCO2403"]
TRIO_WINDOW = 3  # direct left/right neighbour only

trio_hits = rbh_hits[rbh_hits.qseqid.isin(TRIO)]

rows = []
for genome, genome_hits in trio_hits.groupby("genome"):
    gene_positions = get_gene_iterator(FAA_FILES[genome])
    hit_genes = dict(zip(genome_hits.locus_tag, genome_hits.qseqid))
    for hit in genome_hits.itertuples():
        neighbours = find_neighbours(hit.locus_tag, gene_positions, TRIO_WINDOW)
        # another trio gene next to it - a neighbouring copy of the same gene is not counted
        trio_neighbours = [f"{hit_genes[n]}:{n}" for n in neighbours if n in hit_genes and hit_genes[n] != hit.qseqid]
        rows.append({
            "genome": genome,
            "qseqid": hit.qseqid,
            "locus_tag": hit.locus_tag,
            "position": gene_positions[hit.locus_tag],
            "trio_neighbours": ", ".join(trio_neighbours),
            "has_trio_neighbour": "yes" if trio_neighbours else "no",
        })

trio_table = pd.DataFrame(rows).sort_values(["genome", "position"]).reset_index(drop=True)
pd.crosstab(trio_table.qseqid, trio_table.has_trio_neighbour)

has_trio_neighbour,no,yes
qseqid,,
SCO2401,3,127
SCO2402,0,94
SCO2403,2,127


In [13]:
def adjacent_blocks(genome_hits: pd.DataFrame) -> list[set[str]]:
    """Split a genome's trio hits (sorted by position) into runs of genes at most TRIO_WINDOW apart;
    return the set of trio genes in each run."""
    blocks = []
    previous = None
    for hit in genome_hits.itertuples():
        if previous is None or hit.position - previous > TRIO_WINDOW:
            blocks.append(set())
        blocks[-1].add(hit.qseqid)
        previous = hit.position
    return blocks


rows = []
for genome, genome_hits in trio_table.groupby("genome"):
    largest = max(adjacent_blocks(genome_hits), key=len)
    rows.append({
        "genome": genome,
        "trio_genes_found": ", ".join(sorted(genome_hits.qseqid.unique())),
        "largest_block": ", ".join(sorted(largest)),
        "neighbourhood": {3: "trio", 2: "pair"}.get(len(largest), "none"),
    })

trio_summary = pd.DataFrame(rows)
print(f"{len(trio_summary)} genomes with at least one SCO2401-2403 hit")
trio_summary.groupby(["trio_genes_found", "neighbourhood"]).size()

131 genomes with at least one SCO2401-2403 hit


trio_genes_found           neighbourhood
SCO2401                    none              2
SCO2401, SCO2402, SCO2403  trio             94
SCO2401, SCO2403           none              1
                           pair             33
SCO2403                    none              1
dtype: int64

In [15]:
# genomes where the three genes are not all in one block
trio_summary[trio_summary.neighbourhood != "trio"].sort_values(["neighbourhood", "genome"])

,genome,trio_genes_found,largest_block,neighbourhood
2,Peterkaempfera_sp._SMS_1(5)a,SCO2403,SCO2403,none
120,Streptomyces_venezuelae,SCO2401,SCO2401,none
121,Streptomyces_venezuelae_ATCC_10712,SCO2401,SCO2401,none
130,Streptomyces_zaomyceticus,"SCO2401, SCO2403",SCO2403,none
1,Kitasatospora_papulosa,"SCO2401, SCO2403","SCO2401, SCO2403",pair
6,Streptomyces_alboniger,"SCO2401, SCO2403","SCO2401, SCO2403",pair
7,Streptomyces_alfalfae,"SCO2401, SCO2403","SCO2401, SCO2403",pair
16,Streptomyces_bathyalis,"SCO2401, SCO2403","SCO2401, SCO2403",pair
18,Streptomyces_boninensis,"SCO2401, SCO2403","SCO2401, SCO2403",pair
24,Streptomyces_carpaticus,"SCO2401, SCO2403","SCO2401, SCO2403",pair


## 4. Investigate hits without neighbours

`isolated_hits`: for every hit with `has_neighbours == "no"`, the nearest other hit of the same reference group
in that genome and how many genes away it is (NaN = the only hit of its group in that genome).

`show_neighbourhood` lists the `WINDOW` genes on each side of a hit with their full description from the `.faa`
header; `rbh_hit` marks genes that are themselves RBH hits (of either group).

In [6]:
WINDOW = 10


def read_descriptions(faa_file: Path) -> dict[str, str]:
    """Return {locus_tag: description} from the '>Genome|LOCUS_TAG|description|accession' headers."""
    descriptions = {}
    with open(faa_file) as f:
        for line in f:
            if line.startswith(">"):
                fields = line.split("|")
                descriptions[fields[1]] = fields[2]
    return descriptions


def show_neighbourhood(genome: str, locus_tag: str, window: int = WINDOW) -> pd.DataFrame:
    """The window genes on each side of locus_tag, in genome order, with descriptions and RBH hits marked."""
    gene_positions = get_gene_iterator(FAA_FILES[genome])
    descriptions = read_descriptions(FAA_FILES[genome])
    genome_hits = synteny_table[synteny_table.genome == genome]
    hit_genes = dict(zip(genome_hits.locus_tag, genome_hits.qseqid))

    genes = sorted(find_neighbours(locus_tag, gene_positions, window) + [locus_tag], key=gene_positions.get)
    return pd.DataFrame({
        "offset": [gene_positions[g] - gene_positions[locus_tag] for g in genes],
        "locus_tag": genes,
        "description": [descriptions[g] for g in genes],
        "rbh_hit": [hit_genes.get(g, "") for g in genes],
    })


rows = []
for genome, genome_isolated in synteny_table[synteny_table.has_neighbours == "no"].groupby("genome"):
    gene_positions = get_gene_iterator(FAA_FILES[genome])
    genome_hits = synteny_table[synteny_table.genome == genome]
    for hit in genome_isolated.itertuples():
        others = genome_hits[(genome_hits.ref_group == hit.ref_group) & (genome_hits.locus_tag != hit.locus_tag)]
        distances = {f"{o.qseqid}:{o.locus_tag}": abs(gene_positions[o.locus_tag] - gene_positions[hit.locus_tag])
                     for o in others.itertuples()}
        nearest = min(distances, key=distances.get) if distances else None
        rows.append({
            "ref_group": hit.ref_group,
            "genome": genome,
            "qseqid": hit.qseqid,
            "locus_tag": hit.locus_tag,
            "other_group_hits": len(others),
            "nearest_hit": nearest,
            "genes_away": distances.get(nearest),
        })

isolated_hits = pd.DataFrame(rows).sort_values(["ref_group", "qseqid", "genes_away"]).reset_index(drop=True)
isolated_hits.groupby(["ref_group", "qseqid"]).size()

ref_group  qseqid        
SCO        SCO2407            6
           SCO2439            6
           SCO2440            2
VNZ        vnz_33170-lacI    22
           vnz_33175-araB     1
           vnz_33180-araA     1
           vnz_33185-araD     2
dtype: int64

### SCO2407

In [7]:
sco2407 = isolated_hits[isolated_hits.qseqid.isin(["SCO2407"])]
sco2407

,ref_group,genome,qseqid,locus_tag,other_group_hits,nearest_hit,genes_away
0,SCO,Streptomyces_vilmorinianum,SCO2407,FDM97_RS19630,5,SCO2439:FDM97_RS20105,90.0
1,SCO,Streptomyces_poriferorum,SCO2407,P8A21_RS38345,4,SCO2401:P8A21_RS34285,795.0
2,SCO,Streptomyces_paludis,SCO2407,DVK44_RS35315,3,SCO2440:DVK44_RS30615,922.0
3,SCO,Streptomyces_boninensis,SCO2407,ACNNMR_RS34500,4,SCO2403:ACNNMR_RS23280,2188.0
4,SCO,Streptomyces_bathyalis,SCO2407,G4Z16_RS02255,4,SCO2440:G4Z16_RS24380,4331.0
5,SCO,Streptomyces_hebeiensis,SCO2407,ACU6P3_RS00270,4,SCO2403:ACU6P3_RS29140,5509.0


In [8]:
for hit in sco2407.itertuples():
    print(f"{hit.genome}  {hit.locus_tag}  (nearest: {hit.nearest_hit}, {hit.genes_away} genes away)")
    display(show_neighbourhood(hit.genome, hit.locus_tag))

Streptomyces_vilmorinianum  FDM97_RS19630  (nearest: SCO2439:FDM97_RS20105, 90.0 genes away)


,offset,locus_tag,description,rbh_hit
0,-10,FDM97_RS19570,ATP-binding protein,
1,-9,FDM97_RS19580,bifunctional salicylyl-CoA 5-hydroxylase/oxido...,
2,-8,FDM97_RS19585,carbon-nitrogen hydrolase family protein,
3,-7,FDM97_RS19590,cupin domain-containing protein,
4,-6,FDM97_RS19595,fumarylacetoacetate hydrolase family protein,
5,-5,FDM97_RS19600,maleate cis-trans isomerase family protein,
6,-4,FDM97_RS19605,PaaX family transcriptional regulator,
7,-3,FDM97_RS19610,hypothetical protein,
8,-2,FDM97_RS19620,YbhB/YbcL family Raf kinase inhibitor-like pro...,
9,-1,FDM97_RS19625,nucleoside-triphosphatase,


Streptomyces_poriferorum  P8A21_RS38345  (nearest: SCO2401:P8A21_RS34285, 795.0 genes away)


,offset,locus_tag,description,rbh_hit
0,-10,P8A21_RS38290,SRPBCC family protein,
1,-9,P8A21_RS38295,radical SAM protein,
2,-8,P8A21_RS38300,LysR family transcriptional regulator,
3,-7,P8A21_RS38305,oxidoreductase,
4,-6,P8A21_RS38315,GH25 family lysozyme,
5,-5,P8A21_RS38320,MarR family winged helix-turn-helix transcript...,
6,-4,P8A21_RS38325,hypothetical protein,
7,-3,P8A21_RS38330,multiple monosaccharide ABC transporter substr...,
8,-2,P8A21_RS38335,multiple monosaccharide ABC transporter ATP-bi...,
9,-1,P8A21_RS38340,multiple monosaccharide ABC transporter permease,


Streptomyces_paludis  DVK44_RS35315  (nearest: SCO2440:DVK44_RS30615, 922.0 genes away)


...

,offset,locus_tag,description,rbh_hit
0,-10,ACNNMR_RS34445,FGGY family carbohydrate kinase,
1,-9,ACNNMR_RS34450,sugar ABC transporter permease,
2,-8,ACNNMR_RS34455,ATP-binding cassette domain-containing protein,
3,-7,ACNNMR_RS34460,sugar ABC transporter substrate-binding protein,
4,-6,ACNNMR_RS34465,DeoR/GlpR family DNA-binding transcription reg...,
5,-5,ACNNMR_RS34475,S1 family peptidase,
6,-4,ACNNMR_RS34480,carbohydrate ABC transporter permease,
7,-3,ACNNMR_RS34485,carbohydrate ABC transporter permease,
8,-2,ACNNMR_RS34490,extracellular solute-binding protein,
9,-1,ACNNMR_RS34495,alpha-N-arabinofuranosidase,


Streptomyces_bathyalis  G4Z16_RS02255  (nearest: SCO2440:G4Z16_RS24380, 4331.0 genes away)


,offset,locus_tag,description,rbh_hit
0,-10,G4Z16_RS02205,RNA polymerase sigma factor,
1,-9,G4Z16_RS02210,hypothetical protein,
2,-8,G4Z16_RS02215,DUF5987 family protein,
3,-7,G4Z16_RS02220,GMC family oxidoreductase N-terminal domain-co...,
4,-6,G4Z16_RS02225,aminoglycoside adenylyltransferase domain-cont...,
5,-5,G4Z16_RS02230,GntR family transcriptional regulator,
6,-4,G4Z16_RS02235,ABC transporter ATP-binding protein,
7,-3,G4Z16_RS02240,hypothetical protein,
8,-2,G4Z16_RS02245,glycosyltransferase 87 family protein,
9,-1,G4Z16_RS02250,hypothetical protein,


Streptomyces_hebeiensis  ACU6P3_RS00270  (nearest: SCO2403:ACU6P3_RS29140, 5509.0 genes away)


,offset,locus_tag,description,rbh_hit
0,-10,ACU6P3_RS00220,hypothetical protein,
1,-9,ACU6P3_RS00225,NAD-dependent malic enzyme,
2,-8,ACU6P3_RS00230,AEC family transporter,
3,-7,ACU6P3_RS00235,hypothetical protein,
4,-6,ACU6P3_RS00240,glycine betaine/L-proline transporter ProP,
5,-5,ACU6P3_RS00245,Lrp/AsnC family transcriptional regulator,
6,-4,ACU6P3_RS00250,AbgT family transporter,
7,-3,ACU6P3_RS00255,M20 family metallopeptidase,
8,-2,ACU6P3_RS00260,hypothetical protein,
9,-1,ACU6P3_RS00265,SigE family RNA polymerase sigma factor,


### SCO2439

In [9]:
sco2439 = isolated_hits[isolated_hits.qseqid.isin(["SCO2439"])]
sco2439

,ref_group,genome,qseqid,locus_tag,other_group_hits,nearest_hit,genes_away
6,SCO,Streptomyces_niveus,SCO2439,OG275_RS01590,3,SCO2440:OG275_RS00925,127.0
7,SCO,Streptomyces_rosealbus,SCO2439,ACRAKG_RS14275,4,SCO2407:ACRAKG_RS06195,1585.0
8,SCO,Streptomyces_durmitorensis,SCO2439,M4V62_RS28785,4,SCO2407:M4V62_RS37490,1706.0
9,SCO,Streptomyces_boninensis,SCO2439,ACNNMR_RS12515,4,SCO2440:ACNNMR_RS23265,2080.0
10,SCO,Streptomyces_atratus,SCO2439,C5746_RS34640,0,None,NaN
11,SCO,Streptomyces_halobius,SCO2439,K9S39_RS40770,0,None,NaN


In [10]:
for hit in sco2439.itertuples():
    print(f"{hit.genome}  {hit.locus_tag}  (nearest: {hit.nearest_hit}, {hit.genes_away} genes away)")
    display(show_neighbourhood(hit.genome, hit.locus_tag))

Streptomyces_niveus  OG275_RS01590  (nearest: SCO2440:OG275_RS00925, 127.0 genes away)


,offset,locus_tag,description,rbh_hit
0,-10,OG275_RS01540,hypothetical protein,
1,-9,OG275_RS01545,hypothetical protein,
2,-8,OG275_RS01550,cold-shock protein,
3,-7,OG275_RS01555,DEAD/DEAH box helicase,
4,-6,OG275_RS01560,SCO5918 family protein,
5,-5,OG275_RS01565,hypothetical protein,
6,-4,OG275_RS01570,endonuclease,
7,-3,OG275_RS01575,hypothetical protein,
8,-2,OG275_RS01580,SpoIIE family protein phosphatase,
9,-1,OG275_RS01585,DUF6351 family protein,


Streptomyces_rosealbus  ACRAKG_RS14275  (nearest: SCO2407:ACRAKG_RS06195, 1585.0 genes away)


,offset,locus_tag,description,rbh_hit
0,-10,ACRAKG_RS14225,LysR family transcriptional regulator,
1,-9,ACRAKG_RS14230,DUF5713 family protein,
2,-8,ACRAKG_RS14235,pyroglutamyl-peptidase I family protein,
3,-7,ACRAKG_RS14240,EI24 domain-containing protein,
4,-6,ACRAKG_RS14245,M14 family zinc carboxypeptidase,
5,-5,ACRAKG_RS14250,serine hydrolase domain-containing protein,
6,-4,ACRAKG_RS14255,organic hydroperoxide resistance protein,
7,-3,ACRAKG_RS14260,NADP-dependent oxidoreductase,
8,-2,ACRAKG_RS14265,MarR family winged helix-turn-helix transcript...,
9,-1,ACRAKG_RS14270,SCO2400 family protein,


Streptomyces_durmitorensis  M4V62_RS28785  (nearest: SCO2407:M4V62_RS37490, 1706.0 genes away)


...

,offset,locus_tag,description,rbh_hit
0,-10,ACNNMR_RS12465,DUF3145 domain-containing protein,
1,-9,ACNNMR_RS12470,GDSL-type esterase/lipase family protein,
2,-8,ACNNMR_RS12475,aldose epimerase family protein,
3,-7,ACNNMR_RS12480,glycoside hydrolase family 172 protein,
4,-6,ACNNMR_RS12485,GNAT family N-acetyltransferase,
5,-5,ACNNMR_RS12490,ATP-binding protein,
6,-4,ACNNMR_RS12495,organic hydroperoxide resistance protein,
7,-3,ACNNMR_RS12500,NADP-dependent oxidoreductase,
8,-2,ACNNMR_RS12505,MarR family winged helix-turn-helix transcript...,
9,-1,ACNNMR_RS12510,SpoIIE family protein phosphatase,


Streptomyces_atratus  C5746_RS34640  (nearest: None, nan genes away)


,offset,locus_tag,description,rbh_hit
0,-10,C5746_RS34590,S66 peptidase family protein,
1,-9,C5746_RS34595,glycosyltransferase family 39 protein,
2,-8,C5746_RS34600,asparagine synthase (glutamine-hydrolyzing),
3,-7,C5746_RS34605,DUF6204 family protein,
4,-6,C5746_RS34610,DUF5107 domain-containing protein,
5,-5,C5746_RS34615,VOC family protein,
6,-4,C5746_RS34620,class I SAM-dependent methyltransferase,
7,-3,C5746_RS34625,bifunctional phosphatase PAP2/diacylglycerol k...,
8,-2,C5746_RS34630,TIGR03086 family metal-binding protein,
9,-1,C5746_RS34635,IclR family transcriptional regulator,


Streptomyces_halobius  K9S39_RS40770  (nearest: None, nan genes away)


,offset,locus_tag,description,rbh_hit
0,-10,K9S39_RS40710,roadblock/LC7 domain-containing protein,
1,-9,K9S39_RS40715,ATP-binding protein,
2,-8,K9S39_RS40720,substrate-binding domain-containing protein,
3,-7,K9S39_RS40725,methyltransferase domain-containing protein,
4,-6,K9S39_RS40730,aminotransferase class V-fold PLP-dependent en...,
5,-5,K9S39_RS40735,ArsR/SmtB family transcription factor,
6,-4,K9S39_RS40740,SHOCT domain-containing protein,
7,-3,K9S39_RS40745,hypothetical protein,
8,-2,K9S39_RS40760,PepSY domain-containing protein,
9,-1,K9S39_RS40765,IclR family transcriptional regulator domain-c...,


### VNZ araB and araD

In [11]:
vnz_ara = isolated_hits[isolated_hits.qseqid.isin(["vnz_33175-araB", "vnz_33185-araD"])]
vnz_ara

,ref_group,genome,qseqid,locus_tag,other_group_hits,nearest_hit,genes_away
36,VNZ,Streptomyces_erythrochromogenes,vnz_33175-araB,OHA91_RS05465,3,vnz_33180-araA:OHA91_RS05365,16.0
38,VNZ,Streptomyces_canus,vnz_33185-araD,OHT68_RS00520,3,vnz_33175-araB:OHT68_RS02535,356.0
39,VNZ,Streptomyces_ipomoeae,vnz_33185-araD,ACNJZD_RS24120,1,vnz_33170-lacI:ACNJZD_RS14055,1930.0


In [12]:
for hit in vnz_ara.itertuples():
    print(f"{hit.genome}  {hit.locus_tag}  (nearest: {hit.nearest_hit}, {hit.genes_away} genes away)")
    display(show_neighbourhood(hit.genome, hit.locus_tag))

Streptomyces_erythrochromogenes  OHA91_RS05465  (nearest: vnz_33180-araA:OHA91_RS05365, 16.0 genes away)


,offset,locus_tag,description,rbh_hit
0,-10,OHA91_RS05395,hypothetical protein,
1,-9,OHA91_RS05400,ATP-binding protein,
2,-8,OHA91_RS05405,hypothetical protein,
3,-7,OHA91_RS05410,hypothetical protein,
4,-6,OHA91_RS05415,ROK family protein,
5,-5,OHA91_RS05420,xylulokinase,
6,-4,OHA91_RS05425,xylose isomerase,
7,-3,OHA91_RS05435,GH1 family beta-glucosidase,
8,-2,OHA91_RS05450,aldose epimerase family protein,
9,-1,OHA91_RS05455,hypothetical protein,


Streptomyces_canus  OHT68_RS00520  (nearest: vnz_33175-araB:OHT68_RS02535, 356.0 genes away)


,offset,locus_tag,description,rbh_hit
0,-10,OHT68_RS00455,DUF6192 family protein,
1,-9,OHT68_RS00460,hypothetical protein,
2,-8,OHT68_RS00465,hypothetical protein,
3,-7,OHT68_RS00475,Type 1 glutamine amidotransferase-like domain-...,
4,-6,OHT68_RS00480,NF041680 family putative transposase,
5,-5,OHT68_RS00485,glutaredoxin domain-containing protein,
6,-4,OHT68_RS00490,alpha/beta fold hydrolase,
7,-3,OHT68_RS00500,NF041680 family putative transposase,
8,-2,OHT68_RS00505,transposase,
9,-1,OHT68_RS00510,hypothetical protein,


Streptomyces_ipomoeae  ACNJZD_RS24120  (nearest: vnz_33170-lacI:ACNJZD_RS14055, 1930.0 genes away)


,offset,locus_tag,description,rbh_hit
0,-10,ACNJZD_RS24060,cold-shock protein,
1,-9,ACNJZD_RS24065,MerR family transcriptional regulator,
2,-8,ACNJZD_RS24075,hypothetical protein,
3,-7,ACNJZD_RS24080,sigma-70 family RNA polymerase sigma factor,
4,-6,ACNJZD_RS24085,multicopper oxidase family protein,
5,-5,ACNJZD_RS24090,PA domain-containing protein,
6,-4,ACNJZD_RS24095,extracellular solute-binding protein,
7,-3,ACNJZD_RS24100,carbohydrate ABC transporter permease,
8,-2,ACNJZD_RS24105,IS200/IS605 family transposase,
9,-1,ACNJZD_RS24110,RNA-guided endonuclease InsQ/TnpB family protein,


# what can be removed
ACNJZD_RS24120 reason: the only protein in the pathway
